# CS 412 LIAR: transformer runs (milestones 3.3 and 3.4)

**How to run**
1. `Runtime > Change runtime type > T4 GPU`, then Save.
2. `Runtime > Run all`.
3. Approve the Google Drive popup. Results save to `My Drive/CS412_LIAR/results/transformers/`, one JSON per run.

Every run that already has a result file in Drive is skipped. So if Colab disconnects, just reconnect and `Run all` again. It picks up where it stopped.

Run order (most important first), rough T4 times:
- **Phase 1**: text only, 6-way, official split. BERT-base and RoBERTa-base at lr 1e-5 / 2e-5 / 3e-5; BERT-large and RoBERTa-large at 1e-5. About 1 hour.
- **Phase 2**: text + metadata (corrected and raw credit), 6-way, each model at its best lr from Phase 1. About 1.5 hours.
- **Phase 3**: speaker-disjoint split, RoBERTa-base. About 15 min.
- **Phase 4**: binary task, text only, all four models. About 40 min.
- **Phase 5**: leave-one-field-out ablation, RoBERTa-base. About 40 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"

In [ ]:
!pip -q install -U "transformers>=4.46" datasets accelerate kagglehub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/CS412_LIAR'
OUT = f'{DRIVE}/results/transformers'
os.makedirs(OUT, exist_ok=True)
os.makedirs('/content/412/src', exist_ok=True)
os.makedirs('/content/412/data', exist_ok=True)
print('results ->', OUT)

Project code, copied from `src/` in the repo.

In [ ]:
%%writefile /content/412/src/data.py
"""LIAR loading, cleaning, credit-history leakage correction, and splits."""
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATA_DIR = Path(__file__).resolve().parent.parent / "data"

COLS = ["id", "label", "statement", "subject", "speaker", "job", "state", "party",
        "barely_true_c", "false_c", "half_true_c", "mostly_true_c", "pants_fire_c", "context"]
LABELS = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
COUNT_COLS = ["pants_fire_c", "false_c", "barely_true_c", "half_true_c", "mostly_true_c"]
# label -> credit-history column that counts it ("true" has no column in LIAR)
LABEL2COUNT = {"pants-fire": "pants_fire_c", "false": "false_c", "barely-true": "barely_true_c",
               "half-true": "half_true_c", "mostly-true": "mostly_true_c"}
CAT_COLS = ["speaker", "job", "state", "party", "context", "subject"]


def load_split(name: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / f"{name}.tsv", sep="\t", header=None, names=COLS,
                     quoting=3, dtype=str, keep_default_na=False)
    for c in COUNT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    for c in CAT_COLS + ["statement"]:
        df[c] = df[c].fillna("").str.strip()
    df["party"] = df["party"].str.lower()
    df["y"] = df["label"].map(LABEL2ID)
    df["y_bin"] = (df["y"] >= 3).astype(int)  # half-true and above = true-leaning
    df["split"] = name
    return df


def load_all() -> pd.DataFrame:
    return pd.concat([load_split(s) for s in ["train", "valid", "test"]], ignore_index=True)


def correct_credit(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with the current statement's own label removed from its credit history.

    Only subtracts when the matching count is >= 1, so rows whose history was
    collected before the statement was added are left alone.
    """
    out = df.copy()
    for lab, col in LABEL2COUNT.items():
        m = (out["label"] == lab) & (out[col] >= 1)
        out.loc[m, col] -= 1
    return out


def speaker_disjoint(df: pd.DataFrame, seed: int = 412):
    """Re-split all 12.8k statements so no speaker appears in more than one split.

    Target sizes roughly match the official 80/10/10 split.
    """
    groups = df["speaker"].replace("", "__unknown__")
    sizes = groups.value_counts()
    rng = np.random.default_rng(seed)
    # The very largest speakers (e.g. barack-obama, ~600 rows) would swamp a 10% split,
    # so they always stay in train; everyone else is shuffled and assigned greedily.
    big = set(sizes[sizes > 0.02 * len(df)].index)
    order = [s for s in rng.permutation(sizes.index.to_numpy()) if s not in big]
    target = int(0.10 * len(df))
    assign, n_te, n_va = {}, 0, 0
    for s in order:
        if n_te < target:
            assign[s] = "test"; n_te += sizes[s]
        elif n_va < target:
            assign[s] = "valid"; n_va += sizes[s]
        else:
            assign[s] = "train"
    split = groups.map(lambda s: assign.get(s, "train"))
    out = df.copy()
    out["split"] = split.values
    return (out[out.split == "train"].copy(), out[out.split == "valid"].copy(),
            out[out.split == "test"].copy())

In [ ]:
%%writefile /content/412/src/finetune_transformer.py
"""Fine-tune BERT / RoBERTa on LIAR (milestones 3.3 and 3.4). Needs a GPU (Colab T4 is fine).

Examples:
  python finetune_transformer.py --model bert-base-uncased --meta none
  python finetune_transformer.py --model roberta-base --meta all --split disjoint
  python finetune_transformer.py --model roberta-large --meta all --drop credit --lr 1e-5
  python finetune_transformer.py --model roberta-base --meta all --task binary --seed 2

Metadata is serialized as text and prepended to the statement, with the
corrected credit history (own label removed) unless --raw-credit is passed.
Results go to results/transformers/<run-name>.json.
"""
import argparse
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)
from datasets import Dataset

from data import LABELS, correct_credit, load_split, speaker_disjoint

ROOT = Path(__file__).resolve().parent.parent
FIELDS = ["party", "speaker", "job", "state", "context", "subject", "credit"]


def serialize(r, fields):
    bits = []
    if "speaker" in fields: bits.append(f"speaker: {r.speaker.replace('-', ' ')}")
    if "party" in fields: bits.append(f"party: {r.party}")
    if "job" in fields and r.job: bits.append(f"job: {r.job}")
    if "state" in fields and r.state: bits.append(f"state: {r.state}")
    if "context" in fields and r.context: bits.append(f"context: {r.context}")
    if "subject" in fields: bits.append(f"subject: {r.subject.replace(',', ', ')}")
    if "credit" in fields:
        bits.append(f"history: pants-fire {r.pants_fire_c}, false {r.false_c}, barely-true "
                    f"{r.barely_true_c}, half-true {r.half_true_c}, mostly-true {r.mostly_true_c}")
    return " | ".join(bits)


def build_text(df, fields):
    if not fields:
        return df["statement"].tolist()
    return [f"{serialize(r, fields)} || {r.statement}" for r in df.itertuples()]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="roberta-base")
    ap.add_argument("--meta", choices=["none", "all"], default="none")
    ap.add_argument("--drop", default="", help="comma list of metadata fields to ablate")
    ap.add_argument("--raw-credit", action="store_true")
    ap.add_argument("--split", choices=["official", "disjoint"], default="official")
    ap.add_argument("--task", choices=["6way", "binary"], default="6way")
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--grad-accum", type=int, default=1, help="effective batch = bs * grad-accum")
    ap.add_argument("--max-len", type=int, default=160)
    ap.add_argument("--seed", type=int, default=412)
    ap.add_argument("--out-dir", default="", help="where result JSON goes (default results/transformers)")
    ap.add_argument("--keep-checkpoints", action="store_true")
    a = ap.parse_args()

    tr, va, te = (load_split(s) for s in ["train", "valid", "test"])
    if a.split == "disjoint":
        tr, va, te = speaker_disjoint(pd.concat([tr, va, te], ignore_index=True))
    if not a.raw_credit:
        tr, va, te = (correct_credit(d) for d in (tr, va, te))
    fields = [] if a.meta == "none" else [f for f in FIELDS if f not in a.drop.split(",")]
    target = "y" if a.task == "6way" else "y_bin"
    labels = LABELS if a.task == "6way" else ["false-leaning", "true-leaning"]

    tok = AutoTokenizer.from_pretrained(a.model)
    def mk(df):
        ds = Dataset.from_dict({"text": build_text(df, fields), "label": df[target].tolist()})
        return ds.map(lambda b: tok(b["text"], truncation=True, max_length=a.max_len), batched=True)
    dtr, dva, dte = mk(tr), mk(va), mk(te)

    model = AutoModelForSequenceClassification.from_pretrained(a.model, num_labels=len(labels))

    def compute(p):
        pred = p.predictions.argmax(-1)
        return {"acc": accuracy_score(p.label_ids, pred),
                "macro_f1": f1_score(p.label_ids, pred, average="macro")}

    name = (f"{a.model.split('/')[-1]}_{a.meta}{'-' + a.drop if a.drop else ''}"
            f"{'_rawcredit' if a.raw_credit else ''}_{a.split}_{a.task}_lr{a.lr:g}_s{a.seed}")
    args = TrainingArguments(
        output_dir=str(ROOT / "checkpoints" / name), learning_rate=a.lr,
        per_device_train_batch_size=a.bs, gradient_accumulation_steps=a.grad_accum,
        per_device_eval_batch_size=64,
        num_train_epochs=a.epochs, weight_decay=0.01,
        warmup_steps=int(0.1 * a.epochs * -(-len(dtr) // (a.bs * a.grad_accum))),  # 10% warmup; works on transformers 4.x and 5.x
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", save_total_limit=1, seed=a.seed,
        fp16=torch.cuda.is_available(), report_to="none", logging_steps=50)
    trainer = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                      processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=compute)
    train_out = trainer.train()
    val_hist = [{"epoch": h["epoch"], "macro_f1": h["eval_macro_f1"], "acc": h["eval_acc"]}
                for h in trainer.state.log_history if "eval_macro_f1" in h]

    out = trainer.predict(dte)
    pred = out.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(out.label_ids, pred,
                                                    labels=range(len(labels)), zero_division=0)
    res = {"run": name, "args": vars(a), "acc": accuracy_score(out.label_ids, pred),
           "macro_f1": f1_score(out.label_ids, pred, average="macro"),
           "weighted_f1": f1_score(out.label_ids, pred, average="weighted"),
           "per_class": {l: {"p": pr[i], "r": rc[i], "f1": f1[i]} for i, l in enumerate(labels)},
           "val_macro_f1": trainer.state.best_metric, "val_history": val_hist,
           "train_runtime_s": train_out.metrics.get("train_runtime"),
           "test_pred": pred.tolist()}
    d = Path(a.out_dir) if a.out_dir else ROOT / "results" / "transformers"
    d.mkdir(parents=True, exist_ok=True)
    (d / f"{name}.json").write_text(json.dumps(res, indent=2, default=float))
    if not a.keep_checkpoints:
        shutil.rmtree(args.output_dir, ignore_errors=True)
    print(name, f"val_macroF1={res['val_macro_f1']:.4f} test acc={res['acc']:.4f} macroF1={res['macro_f1']:.4f}")


if __name__ == "__main__":
    main()

Get LIAR and check it is byte-identical to the team's copy (same MD5 as `data/` in the repo).
Tries the original UCSB release first, then Kaggle, then `My Drive/CS412_LIAR/archive.zip` (drop the Kaggle zip there if both downloads fail).

In [ ]:
import hashlib, zipfile, io, glob, shutil, urllib.request
MD5 = {'train.tsv': '735058b28189e774d7a7a12334cb8219',
       'valid.tsv': '41e0cf170bbb85bb15eb6d3267c9cd22',
       'test.tsv':  '394cd917733276391ceac6ac7542ebe0'}
D = '/content/412/data'

def ok():
    return all(os.path.exists(f'{D}/{n}') and hashlib.md5(open(f'{D}/{n}','rb').read()).hexdigest() == h
               for n, h in MD5.items())

def from_zip(z):
    for n in MD5:
        m = [x for x in z.namelist() if x.endswith(n)]
        if m: open(f'{D}/{n}', 'wb').write(z.read(m[0]))

if not ok():
    try:
        b = urllib.request.urlopen('https://www.cs.ucsb.edu/~william/data/liar_dataset.zip', timeout=60).read()
        from_zip(zipfile.ZipFile(io.BytesIO(b))); print('UCSB download')
    except Exception as e: print('UCSB failed:', e)
if not ok():
    try:
        import kagglehub
        p = kagglehub.dataset_download('doanquanvietnamca/liar-dataset')
        for n in MD5:
            f = glob.glob(f'{p}/**/{n}', recursive=True)
            if f: shutil.copy(f[0], f'{D}/{n}')
        print('Kaggle download')
    except Exception as e: print('Kaggle failed:', e)
if not ok() and os.path.exists(f'{DRIVE}/archive.zip'):
    from_zip(zipfile.ZipFile(f'{DRIVE}/archive.zip')); print('Drive archive.zip')
assert ok(), 'Data missing or MD5 mismatch. Put the Kaggle archive.zip in My Drive/CS412_LIAR/ and rerun.'
print('LIAR verified: identical to the team copy')

Runner. Skips runs whose result JSON already exists in Drive, and keeps going if one run fails.

In [ ]:
import subprocess, json, glob
SEED = 412
BASE = ['roberta-base', 'bert-base-uncased']
LARGE = ['roberta-large', 'bert-large-uncased']
os.environ['TQDM_DISABLE'] = '1'

def run_name(model, meta='none', drop='', raw=False, split='official', task='6way', lr=2e-5, seed=SEED):
    return (f"{model.split('/')[-1]}_{meta}{'-' + drop if drop else ''}"
            f"{'_rawcredit' if raw else ''}_{split}_{task}_lr{lr:g}_s{seed}")

def run(model, lr, meta='none', drop='', raw=False, split='official', task='6way', seed=SEED):
    name = run_name(model, meta, drop, raw, split, task, lr, seed)
    if os.path.exists(f'{OUT}/{name}.json'):
        print('skip (done):', name); return
    bs = 16 if model in LARGE else 32
    cmd = ['python', 'finetune_transformer.py', '--model', model, '--lr', str(lr), '--bs', str(bs),
           '--meta', meta, '--split', split, '--task', task, '--seed', str(seed), '--out-dir', OUT]
    if drop: cmd += ['--drop', drop]
    if raw: cmd += ['--raw-credit']
    print('\n=== RUN', name, '===', flush=True)
    p = subprocess.Popen(cmd, cwd='/content/412/src', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if any(k in line for k in ("'loss'", 'eval_macro_f1', 'macroF1', 'Error', 'error')):
            print(line, end='', flush=True)
    if p.wait() != 0:
        print('FAILED:', name)

def best_lr(model):
    rs = [json.load(open(f)) for f in glob.glob(f"{OUT}/{model.split('/')[-1]}_none_official_6way_lr*_s{SEED}.json")]
    if not rs: return 1e-5 if model in LARGE else 2e-5
    return max(rs, key=lambda r: r['val_macro_f1'])['args']['lr']

def summary():
    rows = sorted((json.load(open(f)) for f in glob.glob(f'{OUT}/*.json')), key=lambda r: r['run'])
    print(f"{'run':70s} {'val F1':>7s} {'test F1':>7s} {'acc':>6s}")
    for r in rows:
        print(f"{r['run']:70s} {r['val_macro_f1']:7.3f} {r['macro_f1']:7.3f} {r['acc']:6.3f}")

## Phase 1: text only, 6-way, official split

In [ ]:
for m in BASE:
    for lr in [1e-5, 2e-5, 3e-5]:
        run(m, lr)
for m in LARGE:
    run(m, 1e-5)
summary()

## Phase 2: text + metadata, 6-way, official split (corrected and raw credit)

In [ ]:
for m in BASE + LARGE:
    lr = best_lr(m); print(m, 'best lr', lr)
    run(m, lr, meta='all')            # corrected credit
    run(m, lr, meta='all', raw=True)  # raw credit (leaky)
summary()

## Phase 3: speaker-disjoint split, RoBERTa-base

In [ ]:
m = 'roberta-base'; lr = best_lr(m)
run(m, lr, split='disjoint')
run(m, lr, meta='all', split='disjoint')
run(m, lr, meta='all', raw=True, split='disjoint')
summary()

## Phase 4: binary task, text only

In [ ]:
for m in BASE + LARGE:
    run(m, best_lr(m), task='binary')
summary()

## Phase 5: leave-one-field-out ablation, RoBERTa-base, corrected credit

In [ ]:
m = 'roberta-base'; lr = best_lr(m)
for f in ['party', 'speaker', 'job', 'state', 'context', 'subject', 'credit']:
    run(m, lr, meta='all', drop=f)
summary()